# RepoHunter

GitHub 仓库发现与分析流水线：需求清单 → 检索 → Content Filter → 排序报告。

本笔记本只做 import 和调用，函数实现都在 `hunter/` 包里。每个 cell 上面有 md 说明它干什么、进什么、出什么。

## 1. Initialization

import 核心包各模块。密钥从 `.env` 读（见 `hunter/config.py`），不写死在 notebook 里。

In [2]:
from hunter.config import load_skill, MODELS
from hunter.clients import mcp_session
from hunter.pre_filter.query_understanding import Query_Understanding
from hunter.pre_filter.search_repo import Search_Repositories
from hunter.repo_detection import Repo_Detection
from hunter import visual

## 2. Query_Understanding

从用户写的需求清单（keypoints）里抽出 GitHub 搜索查询。keypoints 由用户直接写、原样带回，不经 QU 改动。语言由用户单独填，走检索硬过滤。

输入 skill 文本 + 需求清单，输出含 queries / keypoints 的 dict。

In [3]:
qu_skill = load_skill("query_understanding")

# 用户一条条写的需求清单，就是后面 Content Filter 逐条核对的 keypoints
keypoints = [
    "比较好的 agent 项目，适合写进简历的中小型项目",
    "不要大型项目",
    "必须是多 agent",
]

# 语言走检索硬过滤，最多三个，每格一个，手输会自动纠错归一（pthon->Python），选 Python 或 ipynb 会自动补上另一个
lang_1 = "Python"
lang_2 = None
lang_3 = None
target_languages = [lang for lang in (lang_1, lang_2, lang_3) if lang]

queries = await Query_Understanding(qu_skill, keypoints)
queries

生成 1 路查询，3 条 keypoint
token 消耗 3020（命中95%）


{'queries': [{'q': '(multi-agent OR multi agent OR multiagent)'}],
 'keypoints': ['比较好的 agent 项目，适合写进简历的中小型项目', '不要大型项目', '必须是多 agent']}

## Keypoint_Understanding（独立测试，不进流水线）

把每条 keypoint 各自编译成一句可判定的标准（standard），keypoint 原文照旧不改，standard 是附加字段。每条 keypoint 独立发一次请求、全部并发，互相不干扰，某条调用失败该条 standard 给空串，不拖累其他条。

这个模块还没接进 `Repo_Detection`，先在这单独跑通看编译质量：有没有把模糊的地方说清楚、有没有塞用户没说的条件。

In [4]:
from hunter.config import load_skill
from hunter.pre_filter.keypoint_understanding import Keypoint_Understanding

kp_skill = load_skill("keypoint_understanding")

compiled = await Keypoint_Understanding(kp_skill, keypoints)
for item in compiled:
    print(f"keypoint: {item['keypoint']}")
    print(f"standard: {item['standard']}\n")

编译 3 条 keypoint，token 消耗 2321（命中90%）
keypoint: 比较好的 agent 项目，适合写进简历的中小型项目
standard: 项目功能明确、核心流程可运行且无明显缺陷，规模中小（代码量大致在数千至数万行级别，个人可主导），技术方案能体现 agent 设计思路的算；仅空壳、无法运行、代码过于简单或规模过于庞大的不算，看实际仓库情况掂量

keypoint: 不要大型项目
standard: 项目规模明显庞大（如体积很大、代码量庞大或资源消耗高）的不算，一般规模的算，结合实际掂量

keypoint: 必须是多 agent
standard: 标准：项目架构或代码实现中明确存在多个独立自主的 Agent 实体（各 Agent 有自己的目标、行动或决策逻辑，彼此可协作或对抗）且数量≥2，算。仅支持单 Agent 多工具/多模块、或只在名称里提到“多Agent”但实际无独立多实体运行的，不算。



## 3. Search_Repositories

把 QU 的查询发给 GitHub 搜索，best-match 取 top_k、按 full_name 去重成候选池。用户填的 target_languages 当统一硬过滤。

输入 mcp session + queries + languages，输出去重后的仓库列表（每项 full_name / description / topics）。

In [5]:
async with mcp_session() as session:
    repos = await Search_Repositories(session, queries, languages=target_languages, top_k=30)
repos

[{'full_name': 'TauricResearch/TradingAgents',
  'description': 'TradingAgents: Multi-Agents LLM Financial Trading Framework',
  'topics': ['agent', 'finance', 'llm', 'multiagent', 'trading'],
  'stars': 91198,
  'size': 5192},
 {'full_name': 'aden-hive/hive',
  'description': 'Multi-Agent Harness for Production AI',
  'topics': ['agent',
   'agent-framework',
   'agent-skills',
   'anthropic',
   'automation',
   'autonomous-agents',
   'claude',
   'harness',
   'harness-engineering',
   'human-in-the-loop',
   'openai',
   'python',
   'self-hosted',
   'self-improving'],
  'stars': 10637,
  'size': 19973},
 {'full_name': 'openai/openai-agents-python',
  'description': 'A lightweight, powerful framework for multi-agent workflows',
  'topics': ['agents',
   'ai',
   'framework',
   'harness',
   'llm',
   'openai',
   'python'],
  'stars': 27686,
  'size': 34525},
 {'full_name': 'FoundationAgents/MetaGPT',
  'description': '🌟 The Multi-Agent Framework: First AI Software Company, Towa

## 4. Repo_Detection（单仓库测试版）

`Repo_Detection_test(repo, query=None, log="ReAct")` 只吃一个仓库 URL，跳过 GitHub 检索，直接塞进去跑完整流程：先过 gate 判要不要深挖，判要深挖就用四个只读工具读源码，content 只摆事实产出架构拆解，keypoint 判定走辩论（正反两个立场 agent 带工具并行取证，锚点核验后由裁决者出终判）。逻辑照搬正式 `Repo_Detection` 里的 process_one。

`query` 是一整句自然语言需求（或一个 list），跟 `Repo_Detection` 的 `query` 参数同一套转换规则：整句字符串当唯一一条 keypoint，list 原样当多条 keypoint 用，不传就用一份默认需求清单，本质上和前端一条条填 keypoint 是同一件事。

`log` 控制日志形态：`"ReAct"`（默认）原样打印 `explore_one` 主循环每轮发给 LLM 的新增消息和 LLM 的原始回复，不做任何裁剪或格式修饰，用于评估 agent trajectory；`"Visual"` 是排版整理过的版本，round 之间有分隔线和空行，工具调用逐条独立成行，更适合人眼快速扫。

In [7]:
import asyncio
import os
import re
import httpx
from unittest.mock import patch

from hunter import config
from hunter.config import load_skill, MODELS, CLONE_DIR
from hunter.repo_detection.prefetch import fetch_readme, clean_readme
from hunter.repo_detection.prefetch import fetch_tree, fetch_meta
from hunter.repo_detection import explorer
from hunter.repo_detection.explorer import explore_one
from hunter.repo_detection import _fill_system, _fill_gate, _fill_explore
from hunter.pre_filter.keypoint_understanding import Keypoint_Understanding


def _find_existing_clone(full_name: str) -> str | None:
    """在 CLONE_DIR 下找这个仓库有没有已经克隆好的目录，读 remote url 反查匹配。

    目录名是随机的 repohunter_xxxxxx，认不出对应哪个仓库，只能挨个读 .git/config
    里的 origin url 才知道。remote url 可能带 PAT（https://<PAT>@github.com/...），
    正则把凭证部分剥掉再比对 owner/name，大小写不敏感。

    Args:
        full_name: owner/name。
    Returns:
        已有克隆目录的绝对路径；没找到或目录已被清空返回 None。
    """
    if not os.path.isdir(CLONE_DIR):
        return None
    target = full_name.lower()
    for entry in os.scandir(CLONE_DIR):
        if not entry.is_dir():
            continue
        git_dir = os.path.join(entry.path, ".git")
        if not os.path.isdir(git_dir):
            continue
        try:
            with open(os.path.join(git_dir, "config"), encoding="utf-8", errors="replace") as f:
                cfg = f.read()
        except OSError:
            continue
        m = re.search(r"url\s*=\s*.*github\.com[/:]([^/\s]+/[^/\s]+?)(?:\.git)?\s*$", cfg, re.MULTILINE)
        if m and m.group(1).lower() == target:
            return entry.path
    return None


async def Repo_Detection_test(repo: str, query: str | list[str] | None = None,
                              output_language: str = "简体中文", log: str = "ReAct") -> dict:
    """只测一个仓库的完整 Content Filter，跳过 GitHub 检索，直接塞 repo 进去。

    逻辑照搬 Repo_Detection 里的 process_one：编译 keypoint 标准、预抓 readme/目录树/meta、
    拼 system 和三份 user、调 explore_one 跑 gate → content → 辩论裁决。这个 cell 自带全部
    import，不依赖前面的章节，单独跑就能用。

    CLONE_DIR 下已经克隆过这个仓库就直接复用那份，不重新克隆，省下每次测试都要等 git clone
    的时间；explore_one 内部的 _clone_repo 只在跑这次测试期间被替换，跑完自动还原，不影响
    正式流水线并发跑其它仓库时各自真实克隆。

    Args:
        repo:            仓库 URL 或 owner/name，如 https://github.com/DawnofeL/Mingchao_Agentic_RAG。
        query:           自然语言需求，跟 Repo_Detection 的 query 参数同一套规则：传一整句字符串
                         就当成唯一一条 keypoint，传 list 就当成多条 keypoint 原样用。不传就用
                         一份默认需求清单。
        output_language: LLM 散文字段用的语言，默认简体中文。
        log:             "ReAct" 原样打印 explore_one 主循环每轮发给 LLM 的新增消息和 LLM
                         原始回复，不裁剪不修饰，用于评估 agent trajectory；"Visual" 是整理过
                         排版的日志，round 之间有分隔线和空行，更适合人眼快速扫。见 explore_one
                         的 log 参数说明。默认 ReAct，因为这个测试函数就是给看原生轨迹用的。
    Returns:
        单个仓库的 explore_one 结果 dict。
    """

    # URL 或 owner/name 都收，剥掉前缀和末尾斜杠，取最后两段当 owner/name
    full_name = repo.rstrip("/").replace("https://github.com/", "").replace("http://github.com/", "")
    owner, name = full_name.split("/")[-2:]
    full_name = f"{owner}/{name}"

    # query 转 keypoints 的规则和 Repo_Detection 一致：一整句字符串当唯一一条，list 原样用
    if isinstance(query, str):
        keypoints = [query]
    elif query:
        keypoints = list(query)
    else:
        keypoints = [
            "比较好的 agent 项目，适合写进简历的中小型项目",
            "不要大型项目",
            "必须是多 agent",
        ]

    # 编译每条 keypoint 的判定标准，和正式 Repo_Detection 一样，拼进 gate 和辩论 prompt
    standards = {}
    if keypoints:
        compiled = await Keypoint_Understanding(load_skill("keypoint_understanding"), keypoints)
        standards = {c["keypoint"]: c["standard"] for c in compiled}

    # 六个 skill 和 Repo_Detection 里加载的是同一批
    header_md = load_skill("system_header")
    gate_md = load_skill("skip_gate")
    explore_md = load_skill("content_filter")
    advocate_md = load_skill("advocate")
    skeptic_md = load_skill("skeptic")
    adjudicate_md = load_skill("adjudicate")
    model = MODELS["content_filter"]

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {config.GITHUB_PAT}", "Accept": "application/vnd.github+json"},
        timeout=30.0,
    ) as client:

        # 预抓三样，和 process_one 一模一样
        raw, tree, (stars, size) = await asyncio.gather(
            fetch_readme(client, owner, name),
            fetch_tree(client, owner, name),
            fetch_meta(client, owner, name),
        )
        readme = clean_readme(raw) if raw else ""

        # repo dict 只用得上 full_name/description/topics，测试就手造一个
        repo_dict = {"full_name": full_name, "description": "", "topics": []}
        system = _fill_system(header_md, repo_dict, readme, tree, size)
        gate_user = _fill_gate(gate_md, keypoints, standards)
        explore_user = _fill_explore(explore_md, output_language)

    # 已有克隆就直接复用那个目录，explore_one 内部就不会再真的 git clone 一次；
    # 没有就走 explore_one 原本的真实克隆，patch 只在这次调用期间生效，跑完自动还原
    existing = _find_existing_clone(full_name)
    if existing:
        print(f"[{full_name}] ♻ 复用已有克隆：{existing}")

        async def _skip_clone(fn: str) -> str:
            return existing

        with patch.object(explorer, "_clone_repo", _skip_clone):
            return await explore_one(full_name, system, gate_user, explore_user,
                                     advocate_md, skeptic_md, adjudicate_md,
                                     keypoints, stars, size, model, output_language, log=log,
                                     standards=standards)

    return await explore_one(full_name, system, gate_user, explore_user,
                             advocate_md, skeptic_md, adjudicate_md,
                             keypoints, stars, size, model, output_language, log=log,
                             standards=standards)

In [8]:
# 单仓库测试：直接塞一个 repo 和一句自然语言 query，跳过 GitHub 检索
# log="ReAct" 原样打印每轮发给 LLM 的新增消息和 LLM 原始回复；想看排版友好的版本改成 log="Visual"
result = await Repo_Detection_test(
    "https://github.com/TauricResearch/TradingAgents",
    query="比较好的 agent 项目，适合写进简历的中小型项目，必须是多 agent",
    log="Visual",
)
print("\nkeypoint", result["keypoint_hits"], "/", result["keypoint_total"],
      "| tools", result["tools_used"], "| tokens", result["tokens"])

编译 1 条 keypoint，token 消耗 1239（命中88%）
[TauricResearch/TradingAgents] ♻ 复用已有克隆：/home/levizenith/SednaAI/Repo_Hunter/data/tmp/repohunter_8l3l_gig
▶ 开始Content Filter
Round 1
[Round 1] 💰 累计 prompt 8020(命中7680/未命中340, 命中96%) + 输出 388 = 8408 tok
→ read_file({"path": "pyproject.toml"})
→ read_file({"path": "main.py"})
→ read_file({"path": "tradingagents/default_config.py"})
→ read_file({"path": "tradingagents/__init__.py"})
read_file 返回:
1	[build-system]
2	requires = ["setuptools>=61.0"]
3	build-backend = "setuptools.build_meta"
4	
5	[project]
6	name = "tradingagents"
7	version = "0.3.1"
8	description = "TradingAgents: Multi-Agents LLM Financial Trading Framework"
9	readme = "README.md"
10	requires-python = ">=3.10"
11	dependencies = [
12	    "langchain-core>=0.3.81",
13	    "backtrader>=1.9.78.123",
14	    "langchain-anthropic>=0.3.15",
15	    "langchain-experimental>=0.3.4",
…还有 74 行
read_file 返回:
1	from tradingagents.default_config import DEFAULT_CONFIG
2	from tradingagents.graph.trading_gr

In [ ]:
# 手动清空克隆缓存：分析用的浅克隆都留在项目 data/tmp 下，跑完不自动删，方便接着深入看
# 想腾空间或换一批仓库前，跑这个 cell 清掉
from hunter.config import clear_clone_dir, CLONE_DIR

clear_clone_dir()
print("已清空", CLONE_DIR)

## 5. 缓存与成本统计

把各阶段累计的 token 和缓存命中汇成一张表：调用数、输入 token、命中、未命中、输出 token、命中率。

In [6]:
visual.cost_table()

,阶段,调用数,输入token,命中,未命中,输出token,命中率
0,query_understanding,1,1864,1792,72,483,96.1%
1,coarse_filter,30,59361,57216,2145,13085,96.4%
2,content_filter,10,109491,91520,17971,3214,83.6%
3,合计,41,170716,150528,20188,16782,88.2%


## 测试

## 四个只读工具试跑（本地克隆）

改 `TARGET` 换目标仓库，克隆 cell 重复运行不会重复克隆，换了仓库才重新克隆。四个工具 cell 的参数都提在各自开头，改完重跑那格即可；超长输出统一截断显示。

In [3]:
import os
from hunter.repo_detection.explorer import _clone_repo
from hunter.repo_detection.agent_tools import list_tree, read_file, grep_code, glob_files

TARGET = "DawnofeL/Mingchao_Agentic_RAG"




# 输出太长会刷屏，统一截断显示
def clip(s, n=2000):
    s = str(s)
    return s if len(s) <= n else s[:n] + f"\n…（截断，共 {len(s)} 字）"

In [5]:
# 想测哪个仓库改这里



# 同一个仓库克隆过就复用，目录还在才算数；换 TARGET 会重新克隆
_roots = globals().get("_roots", {})
if TARGET not in _roots or not os.path.isdir(_roots[TARGET]):
    _roots[TARGET] = await _clone_repo(TARGET)
root = _roots[TARGET]
print("root =", root)

root = /home/levizenith/SednaAI/RepoHunter/data/tmp/repohunter_q2jzbrf1


In [6]:
# list_tree：列某一层目录。真实流程里根目录树由 system 直接给，这个工具是往更深的子目录挖
tree_path = "rag/graph"

print(clip(await list_tree(root, tree_path)))

nodes/                      [dir]
__init__.py                 [file, 0B]
build.py                    [file, 5886B]
citation_check.py           [file, 2621B]
state.py                    [file, 6373B]
stream.py                   [file, 3395B]


In [7]:
# glob_files：按文件名 pattern 跨整棵树找文件，glob_path 填子目录可缩小范围
glob_pattern = "**/*.py"
glob_path = ""

print(clip(await glob_files(root, glob_pattern, glob_path)))

server/schemas.py
server/app.py
server/__init__.py
output/graph_viz/render_graphs.py
tools/vectorization/__init__.py
tools/vectorization/vectorization.py
tools/pdf_slice/ming_volume_slice.py
tools/pdf_slice/two_level_slice.py
tools/pdf_slice/__init__.py
tools/pdf_slice/langchain_recursive_slice.py
tools/__init__.py
tools/milvus/milvus.py
tools/milvus/__init__.py
app.py
skills/mingchao_people_timeline_builder/scripts/script_chunk_extraction.py
skills/mingchao_people_timeline_merger/scripts/script_extract_keys.py
skills/mingchao_people_timeline_merger/scripts/script_filter_by_key.py
skills/mingchao_people_timeline_builder/scripts/validate_kg.py
skills/mingchao_people_timeline_builder/scripts/script_incremental_merge.py
skills/mingchao_people_evaluation/people_loader.py
skills/mingchao_people_evaluation/self_check.py
skills/mingchao_llm_assessment/extractor.py
skills/mingchao_llm_assessment/self_check.py
skills/mingchao_people_timeline_complier/scripts/script_find_candidates.py
skills/min

In [8]:
# grep_code：正则搜代码。mode 三选一：files_with_matches 只回文件名 / content 带行号匹配行 / count 每文件计数
grep_pattern = "class "
grep_mode = "content"
grep_limit = 10

print(clip(await grep_code(root, grep_pattern, output_mode=grep_mode, head_limit=grep_limit)))

server/schemas.py:4:class HistoryMessage(BaseModel):
server/schemas.py:9:class ChatRequest(BaseModel):
server/app.py:40:class _SSECapture:
output/graph_viz/0_combined.mmd:68:    class qu,single_people,single_timeline,single_direct,orchestrate topStyle
output/graph_viz/0_combined.mmd:69:    class orchestrator,worker,synthesize orchStyle
output/graph_viz/0_combined.mmd:70:    class p_first,p_exec,p_judge,p_retry,p_final,p_partial peopleStyle
output/graph_viz/0_combined.mmd:71:    class t_first,t_exec,t_judge,t_partial timelineStyle
output/graph_viz/0_combined.mmd:72:    class v_retrieve vectorStyle
rag/retrieval/chunk_rrf.py:68:class _FakeEntity:
rag/retrieval/chunk_rrf.py:78:class _FakeHit:
...（共 16 行命中，只显示前 10 行）


In [9]:
# grep 玩法 1，正则捕获结构：一把抓出所有 async 函数定义，模型靠这招不读文件就摸清一个模块有哪些入口
print(clip(await grep_code(root, r"async def \w+", output_mode="content", head_limit=15), 1200))

server/app.py:69:async def startup_progress() -> EventSourceResponse:
server/app.py:72:    async def _gen():
server/app.py:109:async def chat(req: ChatRequest) -> EventSourceResponse:
server/app.py:141:    async def _gen():


In [10]:
# grep 玩法 2，多词并联（|）加大小写不敏感：一次搜遍技术栈关键词，判「用没用向量检索」这类 keypoint 就靠它
print(clip(await grep_code(root, r"milvus|bge|rerank", ignore_case=True, output_mode="files_with_matches"), 1200))

README.md
Project_Guide.md
requirements.txt
INSTALL.md
output/rag_evaluation/vector_eval.html
web/index.html
app.py
tools/milvus/milvus.py
tools/milvus/__init__.py
tools/vectorization/vectorization.py
data/eval_qna/people/people_eval_50_TestResults_agentic.json
data/eval_qna/timeline/timeline_eval_50_TestResults_agentic.json
rag/retrieval/chunk_rrf.py
rag/config/settings.py
rag/retrieval/milvus_lite_setup.py
rag/graph/stream.py
rag/graph/nodes/route_task.py
data/eval_qna/hallu_test/hallu_test_100_TestResults_agentic.json
rag/agent/rag.py
rag/agent/modes/vector_mode.py


In [11]:
# grep 玩法 3，glob 限定范围 + context 带上下文：只搜 rag/ 下的 py，命中行前后各带 1 行，直接看到建图代码怎么写
print(clip(await grep_code(root, r"add_node", path="rag", glob="*.py", output_mode="content", context=1, head_limit=12), 1500))

rag/graph/build.py-53-
rag/graph/build.py:54:    builder.add_node("retrieve", _Retrieve_Node)
rag/graph/build.py-55-    builder.add_edge(START, "retrieve")
--
rag/graph/build.py-149-
rag/graph/build.py:150:    builder.add_node("qu",              _QU_Node)
rag/graph/build.py:151:    builder.add_node("single_people",   _Single_People_Node)
rag/graph/build.py:152:    builder.add_node("single_timeline", _Single_Timeline_Node)
rag/graph/build.py:153:    builder.add_node("single_direct",   _Single_Direct_Node)
rag/graph/build.py:154:    builder.add_node("orchestrate",     _Orchestrate_Node)
rag/graph/build.py-155-
--
...（共 33 行命中，只显示前 12 行）


In [12]:
# grep 玩法 4，count 模式找热点：哪个文件提 llm 最多，哪里就是调模型的核心地带，模型用它决定精读哪个文件
print(clip(await grep_code(root, r"llm", ignore_case=True, output_mode="count", head_limit=12), 800))

server/app.py:9
README.md:16
output/llm_evaluation/eval_100_Eval_vector.html:6
output/llm_evaluation/eval_100_Eval_agentic.html:6
output/llm_evaluation/timeline_eval_50_Eval_agentic.html:6
output/llm_evaluation/hallu_test_100_Eval_agentic.html:6
output/llm_evaluation/people_eval_50_Eval_vector.html:6
output/llm_evaluation/timeline_eval_50_Eval_vector.html:6
output/llm_evaluation/people_eval_50_Eval_agentic.html:6
Project_Guide.md:65
rag/retrieval/people_store.py:5
rag/retrieval/chunk_rrf.py:1
...（共 59 行命中，只显示前 12 行）


In [13]:
# read_file：读文件带行号，可用 offset/limit 分段。read_cache 记已读，同一段重复读会提示已读过
file_path = "README.md"

read_cache = globals().get("read_cache", set())
print(clip(await read_file(root, read_cache, file_path)))

1	# Mingchao Agentic RAG  项目总览 & 快速启动
2	
3	本项目为基于小说《明朝那些事儿》前三卷内容构建的 **Agentic RAG 系统**，目标是**召回覆盖绝大多数纯向量检索无法处理的问题类型**，**包括但不限于**枚举（明朝有哪些开国功臣封了公爵）、依赖（先查出一批事件，再逐项追问各自的核心人物）、指代（「他的儿子是谁」中的多轮指代还原）等等。项目有完整的前端本地网页，也可以使用  [`Agentic_RAG_Test.ipynb`](Agentic_RAG_Test.ipynb)  进行代码运行测试。
4	
5	本项目从开始设计到完成历时两个月有余，期间踩过许多坑，进行过无数次设计优化，有着许多巧思设计，非常感谢您愿意花时间阅读！
6	
7	[Mingchao Agentic RAG  项目总览 & 快速启动](#mingchao-agentic-rag-项目总览-快速启动)
8	
9	- [项目设计速览](#项目设计速览)
10	- [完整交互式流程图](#完整交互式流程图)
11	- **[快速启动](#快速启动)**
12	  - [方式一：手动运行以下所有命令](#方式一手动运行以下所有命令)
13	  - [方式二：Coding Agent 一句话搞定](#方式二coding-agent-一句话搞定)
14	- **[快速卸载](#快速卸载)**
15	- [项目文件目录结构](#项目文件目录结构)
16	- [License](#license)
17	
18	## 项目设计速览
19	
20	> [!TIP]
21	>
22	> **❗️以下所有速览的详细内容请见 [`Project_Guide.md`](Project_Guide.md)，包括但不限于完整的设计思路，踩坑总结，以及对应优化等。**
23	
24	- **双模式**：
25	
26	  - `Vector` 为纯代码检索路径，无 LLM 调用，不读对话历史，拥有极低延迟。
27	
28	  - `Agentic` 在检索前执行意图识别与问题拆解，由 LLM 负责规划，代码负责校验与执行编排。
29	
30	- **向量知识库**：《明朝那些事儿》原文按语义边界切分后，以 **BGE-M3** 做 **dense + sparse** 双路编码存

## 看辩论三方实际塞给 LLM 的完整提示词

只拼提示词、不调 LLM。每个立场 worker 只管**一条** keypoint（几条 keypoint 就并行几对正反 worker，keypoint 由代码绑定回结果，不靠模型照抄）。填一条 keypoint、一份 dissection、stars、size，用 `_fill_stance` / `_fill_adjudicate` 把正方、反方、裁决三份 user 拼出来原样打印。三方共用的 system(仓库资料页)另有 `_fill_system`。工具的参数规格不在提示词文本里，作为 API 的 `tools` 参数单独发，见 `TOOL_SCHEMAS`。

In [ ]:
from hunter.config import load_skill
from hunter.repo_detection.debate import _fill_stance, _fill_adjudicate

# ── 手填这几样，就是提示词里各占位的真值 ──
keypoint = "必须是多 agent"          # 正反方和裁决都只管这一条
stars = 42000
size = 18944          # KB
output_language = "简体中文"

# dissection：content 出的架构拆解，立场 worker 拿它当取证起点(这里贴一份精简示例)
dissection = {
    "purpose": "一个多智能体 LLM 金融交易框架，多个 agent 协作产出买卖决策。",
    "tech_stack": "Python；LangGraph 编排；LangChain 接多家 LLM。",
    "key_designs": [
        {"name": "多空辩论加风险辩论", "detail": "看多看空研究员正反辩论，风控三方再辩，组合经理收口。",
         "where": "tradingagents/agents/researchers/bull_researcher.py:create_bull_researcher"},
    ],
    "architecture": "1. 分析师收集信息。\n\n2. 研究员辩论。\n\n3. 风控辩论后出决策。",
}

# ── 正方 worker 的 user(单条 keypoint) ──
adv_user = _fill_stance(load_skill("advocate"), keypoint, dissection, stars, size, output_language)
print("=" * 30, "正方 advocate user", "=" * 30)
print(adv_user)

In [ ]:
# ── 反方 worker 的 user ──（跟正方共用 _fill_stance，只换 skill，同样单条 keypoint）
ske_user = _fill_stance(load_skill("skeptic"), keypoint, dissection, stars, size, output_language)
print("=" * 30, "反方 skeptic user", "=" * 30)
print(ske_user)

In [ ]:
# ── 裁决 user ──（双方 case 手填一份示例，实际由正反方跑完 + 锚点审计后得到；裁决只看这一条 keypoint）
adv_case = {"keypoint": keypoint, "evidence": "agents 目录多个角色协作。",
            "where": "tradingagents/agents/researchers/bull_researcher.py:create_bull_researcher", "searched": ""}
ske_case = {"keypoint": keypoint, "evidence": "没找到反证。", "where": "", "searched": "确为多 agent"}

adj_user = _fill_adjudicate(load_skill("adjudicate"), keypoint, stars, size, adv_case, ske_case, output_language)
print("=" * 30, "裁决 adjudicate user", "=" * 30)
print(adj_user)